# Task 3: Correlation Between News Sentiment and Stock Returns

### Objective

Analyze whether financial news sentiment has a measurable relationship with stock market daily returns using Natural Language Processing (NLP) and correlation analysis.

## 1. Install NLTK Resources

In [4]:
#import nltk
#nltk.download('vader_lexicon')

## 2. Import Required Libraries

In [5]:
import pandas as pd
import matplotlib.pyplot as plt

from nltk.sentiment import SentimentIntensityAnalyzer
from pandas.tseries.offsets import BDay
df = pd.read_csv("../data/raw/news.csv")
stock_df = pd.read_csv("../data/raw/AAPL.csv")

## 3. Initialize Sentiment Analyzer

### VADER Sentiment Model

In [6]:
sia = SentimentIntensityAnalyzer()

## 4. Compute Sentiment Scores

### Generate Compound Sentiment Scores for Headlines

In [ ]:
df["sentiment"] = df["headline"].apply(
    lambda x: sia.polarity_scores(str(x))["compound"]
)

## 5. Handle Weekend News
### Align Weekend News to the Next Trading Day

Financial markets remain closed on weekends.
To improve alignment with stock prices, weekend news is shifted to the next business day.

In [ ]:
df["aligned_date"] = df["date"]

df.loc[
    df["date"].dt.weekday >= 5,
    "aligned_date"
] = df["date"] + BDay(1)

## 6. Aggregate Daily Sentiment
Compute Average Daily Sentiment per Stock

In [ ]:
daily_sentiment = (
    df.groupby(
        ["stock", df["aligned_date"].dt.date]
    )["sentiment"]
    .mean()
    .reset_index()
)

### Rename Date Column for Clarity

In [ ]:
daily_sentiment.rename(
    columns={"aligned_date": "date"},
    inplace=True
)

In [ ]:
merged = pd.merge(
    daily_sentiment,
    stock_df,
    left_on="date",
    right_on="Date"
)

## 7. Calculate Daily Stock Returns
### Compute Percentage Daily Returns

In [ ]:
stock_df["daily_return"] = (
    stock_df["Adj Close"].pct_change() * 100
)

### Convert Stock Dates to Datetime Format


In [ ]:
stock_df["Date"] = pd.to_datetime(
    stock_df["Date"]
).dt.date

## 8. Merge News and Stock Data
### Combine Sentiment Data with Price Data

In [ ]:
plt.scatter(
    merged["sentiment"],
    merged["daily_return"]
)

plt.xlabel("Sentiment")
plt.ylabel("Daily Return")
plt.title(f"Correlation: {correlation:.2f}")

plt.show()

## 9. Correlation Analysis
### Measure Relationship Between Sentiment and Returns

In [ ]:
correlation = merged["sentiment"].corr(
    merged["daily_return"]
)

print(
    f"Correlation between sentiment and returns: "
    f"{correlation:.4f}"
)

## 10. Data Visualization
### Scatter Plot: Sentiment vs Daily Return

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    merged["sentiment"],
    merged["daily_return"]
)

plt.xlabel("Sentiment Score")
plt.ylabel("Daily Return (%)")

plt.title(
    f"Sentiment vs Daily Return "
    f"(Correlation = {correlation:.2f})"
)

plt.grid(True)

plt.show()

## Interpretation
